# EyeCNN Training — Drowsiness Detection Hybrid Classifier
Trains a lightweight CNN to classify eye state (OPEN vs CLOSED)
using crops extracted from YawDD and NTHU-DD datasets.

In [ ]:
# Cell 1 — Setup
!pip install torch torchvision ai-edge-torch tqdm -q

import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)
from pathlib import Path
import zipfile
from tqdm.notebook import tqdm
import random
import cv2

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — training will be slow on CPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Cell 2 — Load dataset
dataset_zip = "dataset.zip"
if not Path("data").exists():
    with zipfile.ZipFile(dataset_zip, "r") as zf:
        zf.extractall(".")
    print(f"Extracted {dataset_zip}")

data_dir = Path("data/extracted")
open_files = sorted((data_dir / "open").glob("*.png"))
closed_files = sorted((data_dir / "closed").glob("*.png"))

print(f"Open samples:   {len(open_files)}")
print(f"Closed samples: {len(closed_files)}")

images, labels = [], []
for f in open_files:
    img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        images.append(img.astype(np.float32) / 255.0)
        labels.append(0)
for f in closed_files:
    img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        images.append(img.astype(np.float32) / 255.0)
        labels.append(1)

images = np.array(images)
labels = np.array(labels)
print(f"Shape: {images.shape}, dtype: {images.dtype}")
print(f"Class 0 (OPEN):   {np.sum(labels == 0)}")
print(f"Class 1 (CLOSED): {np.sum(labels == 1)}")

fig, axes = plt.subplots(4, 4, figsize=(8, 4))
idxs = random.sample(range(len(images)), 16)
for ax, idx in zip(axes.flat, idxs):
    ax.imshow(images[idx], cmap="gray")
    ax.set_title("OPEN" if labels[idx] == 0 else "CLOSED", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 3 — Split and augment
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

X_train, X_val, y_train, y_val = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"Train: {len(X_train)} | Val: {len(X_val)}")


class EyeCropDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        img = torch.from_numpy(img).unsqueeze(0)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        if self.transform:
            img = self.transform(img)
        return img, label


train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])
val_transform = transforms.Compose([
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_dataset = EyeCropDataset(X_train, y_train, transform=train_transform)
val_dataset = EyeCropDataset(X_val, y_val, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False,
                         num_workers=2, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# Cell 4 — Train
from training.eye_cnn import EyeCNN
import torch.nn as nn
import torch.optim as optim

model = EyeCNN().to(device)
model.count_parameters()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

num_epochs = 30
patience = 5
best_val_loss = float("inf")
patience_counter = 0
best_path = "best_eye_cnn.pth"

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}",
                                leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        total += targets.size(0)
        correct += (preds == targets).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_total += targets.size(0)
            val_correct += (preds == targets).sum().item()

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    scheduler.step()

    print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_path)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"Training complete. Best model saved to {best_path}")

In [ ]:
# Cell 5 — Evaluate
model.load_state_dict(torch.load(best_path, map_location=device,
                                  weights_only=True))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds, zero_division=0)
rec = recall_score(all_targets, all_preds, zero_division=0)
f1 = f1_score(all_targets, all_preds, zero_division=0)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

cm = confusion_matrix(all_targets, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["OPEN", "CLOSED"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix — Validation Set")
plt.show()

# Show misclassified examples
mis_idx = np.where(all_preds != all_targets)[0]
print(f"Misclassified: {len(mis_idx)} / {len(all_targets)}")

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
axes = axes.flat
for i, ax in enumerate(axes):
    if i < min(len(mis_idx), 16):
        idx = mis_idx[i]
        img = X_val[idx]
        ax.imshow(img, cmap="gray")
        ax.set_title(f"True: {'OPEN' if y_val[idx] == 0 else 'CLOSED'} | "
                     f"Pred: {'OPEN' if all_preds[idx] == 0 else 'CLOSED'}",
                     fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Export to TFLite
!pip install ai-edge-torch -q

import ai_edge_torch
import torch

model.load_state_dict(torch.load(best_path, map_location="cpu",
                                  weights_only=True))
model.eval()

sample_input = (torch.randn(1, 1, 32, 64),)

print("Converting to TFLite ...")
edge_model = ai_edge_torch.convert(model.eval(), sample_input)

tflite_path = "eye_state.tflite"
edge_model.export(tflite_path)

import os
size_kb = os.path.getsize(tflite_path) / 1024
print(f"Exported: {tflite_path} ({size_kb:.1f} KB)")
print("\nDownload it to your project's models/ folder:")
print("  from google.colab import files")
print("  files.download('eye_state.tflite')")